## Complete Stock Sector Analysis

We will make all corrections and follow the proper route here to produce the sector_data and the sector_data_engineered.

In [7]:
import pandas as pd
import pandas_ta as ta
import numpy as np
import os
import datetime as dt

import pandas as pd

def prepare_data(data: pd.DataFrame,drop_items: list[str] | None = None) -> pd.DataFrame:
    if drop_items is None:
        drop_items = ["high_price", "low_price"]
    return data.drop(columns=drop_items)

## create daily marktet capitalization

def stock_returns_percent(data: pd.DataFrame) -> pd.DataFrame:
    data['returns_pct']=data['close_price'].pct_change()*100
    return data


def get_market_cap(ticker: str, all_stocks: pd.DataFrame):
    return all_stocks.loc[
        all_stocks['symbol'] == ticker,
        'market_cap'
    ].iloc[0]

## get shares outstanding. We will use this to estimate the daily market cap for each stock
## This is a better way to apply weighting rather than the use of one constant market cap which would have been different in previous years.
## NOTE: There are disadvantages to this. We will include them in our limitations. We are only using it because daily market cap is not available for now.
def get_shares_outstanding(ticker:str,all_stocks:pd.DataFrame):
    return all_stocks.loc[
        all_stocks['symbol'] == ticker,
        'shares_outstanding'
    ].iloc[0]

## apply in data frame to create daily market_cap
def create_daily_market_cap(stock_data:pd.DataFrame,shares_outstanding:float)->pd.DataFrame:
    stock_data['daily_market_cap']=stock_data['close_price']*shares_outstanding
    return stock_data

def complete_daily_returns_weight(sector_stocks_dict: dict) -> dict:
    # Concatenate all dataframes, keeping only date + daily_market_cap
    combined = pd.concat(
        [df[['trade_date', 'daily_market_cap']] for df in sector_stocks_dict.values()],
        ignore_index=True
    )

    # Sum market cap per date across all stocks
    daily_mkt_cap_totals = combined.groupby('trade_date')['daily_market_cap'].sum()

    # Count how many stocks contributed to each date
    stock_counts = combined.groupby('trade_date')['daily_market_cap'].count()

    # Now go back through each stock's dataframe and attach its weight + coverage count
    for ticker, df in sector_stocks_dict.items():
        df['sector_mkt_cap_total'] = df['trade_date'].map(daily_mkt_cap_totals)
        df['daily_returns_weight'] = df['daily_market_cap'] / df['sector_mkt_cap_total']
        df['n_stocks_contributing'] = df['trade_date'].map(stock_counts)
        df['weighted_stock_return'] = df['returns_pct'] * df['daily_returns_weight']

    return sector_stocks_dict

def complete_weighted_sector_stock_return(sector_stocks_dict: dict, sector_df: pd.DataFrame) -> pd.DataFrame:
    sector_stocks_dict = complete_daily_returns_weight(sector_stocks_dict)

    combined = pd.concat(
        [df[['trade_date', 'weighted_stock_return', 'n_stocks_contributing']] for df in sector_stocks_dict.values()],
        ignore_index=True
    )

    daily_totals = combined.groupby('trade_date')['weighted_stock_return'].sum()
    coverage = combined.groupby('trade_date')['n_stocks_contributing'].first()  # same value across all rows for a given date

    sector_df['weighted_sector_returns'] = sector_df['date'].map(daily_totals)
    sector_df['n_stocks_contributing'] = sector_df['date'].map(coverage)

    return sector_df


# Get Sector Market Capitalization. The total for the sector
def get_sector_mkt_cap(all_stocks:pd.DataFrame, stocks_list:list[str]):
    tot_mkt_cap=0
    for stock in stocks_list:
        mkt_cap=get_market_cap(stock,all_stocks)
        tot_mkt_cap+=mkt_cap
    return tot_mkt_cap


## Create the sector index
def create_sector_index(sector_df: pd.DataFrame, start_base:int | None = None) -> pd.DataFrame:
    if start_base is None:
        start_base = 100
    sector_returns = sector_df['weighted_sector_returns'].tolist()

    index_values = []
    for returns in sector_returns:
        start_base = start_base * (1 + (returns / 100))
        index_values.append(start_base)

    sector_df['sector_index'] = index_values
    return sector_df

## Aggregate sector volume
def agg_stock_volumes(sector_folder: str, sector_df: pd.DataFrame) -> pd.DataFrame:
    stock_volume_dict = {}
    for csv_data in os.listdir(sector_folder):
        try:
            ticker = os.path.splitext(csv_data)[0].split('_')[0]
            data = pd.read_csv(os.path.join('stock_data', f'{ticker}.csv')) ## Picking the data from stock_data folder
            data['trade_date'] = pd.to_datetime(data['trade_date'], format='%Y-%m-%d', errors='coerce')
            stock_volume_dict[ticker] = data[['trade_date', 'volume']]
        except Exception as e:
            print('Error: ', e)

    # Combine all stocks' date/volume pairs into one long table
    combined = pd.concat(stock_volume_dict.values(), ignore_index=True)

    # Sum volume per date across all stocks
    daily_volume = combined.groupby('trade_date')['volume'].sum()

    # Align onto sector_df by matching date
    sector_df['sector_volume'] = sector_df['date'].map(daily_volume)
    return sector_df


def lag_returns(data: pd.DataFrame,lag_items: list[int] | None = None) -> pd.DataFrame:
    if lag_items is None:
        lag_items=[1]
    for lag in lag_items:
        data[f'lag_return{lag}']=data['returns_pct'].shift(lag)
    return data
    
def moving_averages(data: pd.DataFrame,averaging_items: list[int] | None = None) -> pd.DataFrame:
    if averaging_items is None:
        averaging_items=[5]
    for avg in averaging_items:
        data[f'stock_MA_{avg}']=data['close_price'].rolling(avg).mean()
    return data

def rolling_volatility(data: pd.DataFrame,volatility_items: list[int] | None = None) -> pd.DataFrame:
    if volatility_items is None:
        volatility_items=[20]
    for vol in volatility_items:
        data[f'returns_volatility_{vol}']=data['returns_pct'].rolling(vol).std()
    return data

def relative_strength_index(data:pd.DataFrame, rsi_length:int | None=None) -> pd.DataFrame:
     if rsi_length is None:
        rsi_length=14

     data['stock_RSI']=ta.rsi(data['close_price'], length=rsi_length)
     return data

def ma_convergence_divergence(data:pd.DataFrame) -> pd.DataFrame:
     data.ta.macd(close='close_price', append=True)
     data.rename(columns={'MACD_12_26_9':'MACD','MACDh_12_26_9':'MACD_hist','MACDs_12_26_9':'MACD_signal'},inplace=True)
     return data

def volume_moving_average(data: pd.DataFrame,volume_items: list[int] | None = None) -> pd.DataFrame:
    if volume_items is None:
        volume_items=[20]
    for vol in volume_items:
        data[f'stock_volume_MA_{vol}']=data['volume'].rolling(vol).mean()
    return data

def clean_and_save(data:pd.DataFrame,stock_name:str):
     cleaned_data=data.dropna()
     cleaned_data.to_csv(f'feature_engineering/stocks_FE/{stock_name.upper()}_ENGINEERED',index=False)


def sector_data_creation(folder_path):
    all_stocks_df=pd.read_csv('stock_data/All_Stocks_Info')

    for sub_folder in os.listdir(folder_path):
        sub_folder_path = os.path.join(folder_path, sub_folder)
        print(f"create subfolder path: {sub_folder_path}")

        ## CREATE SECTOR DATAFRAME
        # Generate dates from January 1, 2017, to today
        date_series = pd.date_range(start="2017-01-01", end=dt.date.today())

        # Create the DataFrame
        sector_df = pd.DataFrame({"date": date_series})

        ## create a sector column. We wil still automate this
        sector_df['sector']=f'{sub_folder}'
        
        if os.path.isdir(sub_folder_path):  # makes sure it's actually a folder, not a stray file

            ## Retrieve tickers in the folder
            stock_ticker_list=[]
        
            for csv_data in os.listdir(sub_folder_path):
                try:
                    ticker = os.path.splitext(csv_data)[0].split('_')[0] ## Captures the name of the stock
                    stock_ticker_list.append(ticker)
                except Exception as e:
                        print('Error: ', e)

                try:
                    share_outstanding=get_shares_outstanding(ticker=ticker,all_stocks=all_stocks_df)
                except Exception as e:
                    print('Error: ', e)
            
            # ## calculating the sector market capitalzation (we wont be using this)
            # sector_market_cap=get_sector_mkt_cap(all_stocks=all_stocks_df, stocks_list=stock_ticker_list)

            mutated_stock_dict={}
            for csv_data in os.listdir(sub_folder_path):
                try:
                    ticker = os.path.splitext(csv_data)[0].split('_')[0]

                    ## retrieve the data from stock_data folder instead of the engineered data
                    data = pd.read_csv(os.path.join('stock_data', f'{ticker}.csv'))

                    ## Prepare the data
                    print(f"{ticker} preparing data")
                    prepared_data=prepare_data(data,drop_items=["high_price", "low_price"])

                    ## Implement stock returns (%)
                    print(f"{ticker} stock returns")
                    data_with_stock_returns=stock_returns_percent(prepared_data)

                    ## The trade_date in stock_data should be converted to datetime object
                    data_with_stock_returns['trade_date'] = pd.to_datetime(data['trade_date'], format='%Y-%m-%d', errors='coerce')

                    ## pull the shares_outstanding from all_stocks_df
                    share_outstanding=get_shares_outstanding(ticker=ticker, all_stocks=all_stocks_df)

                    ## create daily market cap for stock data
                    daily_market_cap_df=create_daily_market_cap(stock_data=data_with_stock_returns,shares_outstanding=share_outstanding)

                    # ## apply weighted return to stock data
                    # weighted_returns_df=weighted_stock_returns(csv_data=data_with_stock_returns,ticker=ticker,sector_market_cap=sector_market_cap,all_stocks=all_stocks_df)

                    ## add the different weighted stock data to a dictionary
                    mutated_stock_dict[f'{ticker}']=daily_market_cap_df
                    
                except Exception as e:
                    print(f"Error! {ticker}", e)

            # ## get weighted returns column
            # weighted_stock_returns_dict=complete_daily_returns_weight(sector_stocks_dict=mutated_stock_dict) This is redundant

            ## aggregate the weighted stock rerturns into sector dataframe

            weighted_sector_returns_df=complete_weighted_sector_stock_return(sector_stocks_dict=mutated_stock_dict, sector_df=sector_df)

            # ## aggregate the weighted returns to form sector returns in sector dataframe
            # weighted_sector_returns_df=weighted_sector_stock_return(weighted_stocks_returns_dict=mutated_stock_dict,sector_df=sector_df)
            weighted_sector_returns_df2=weighted_sector_returns_df.dropna() ## Drop NAN
            
            ## Create the sector index
            sector_index_df=create_sector_index(sector_df=weighted_sector_returns_df2,start_base=100)

            ## Get sector volume
            # print(f'subfolder path passed to agg_stock_volume: {sub_folder_path}')
            sector_volume_df=agg_stock_volumes(sector_folder=sub_folder_path,sector_df=sector_index_df)

            ## Save final sector data before feature engineering
            sector_volume_df.to_csv(f'{folder_path}/{sub_folder}/{sub_folder}_sector', index=False)

            ## TO BE CONTINUED

# ## we will perform feature engineering here
# def sector_feature_engineering(sector_df:pd.DataFrame) -> 



In [8]:
FOLDER_PATH='feature_engineering/stocks_sectors'

sector_data_creation(folder_path=FOLDER_PATH)

create subfolder path: feature_engineering/stocks_sectors\agriculture
ELLAHLAKES preparing data
ELLAHLAKES stock returns
FTNCOCOA preparing data
FTNCOCOA stock returns
LIVESTOCK preparing data
LIVESTOCK stock returns
OKOMUOIL preparing data
OKOMUOIL stock returns
create subfolder path: feature_engineering/stocks_sectors\conglomerates
CUSTODIAN preparing data
CUSTODIAN stock returns
JOHNHOLT preparing data
JOHNHOLT stock returns
SCOA preparing data
SCOA stock returns
TRANSCORP preparing data
TRANSCORP stock returns
UACN preparing data
UACN stock returns
create subfolder path: feature_engineering/stocks_sectors\construction_real_estate
JBERGER preparing data
JBERGER stock returns
UPDCREIT preparing data
UPDCREIT stock returns
UPDC preparing data
UPDC stock returns
create subfolder path: feature_engineering/stocks_sectors\consumer_goods
BUAFOODS preparing data
BUAFOODS stock returns
CADBURY preparing data
CADBURY stock returns
DANGSUGAR preparing data
DANGSUGAR stock returns
GUINNESS prep

In [9]:
fin_service_df=pd.read_csv('feature_engineering/stocks_sectors/financial_services/financial_services_sector')

In [10]:
fin_service_df.shape

(2263, 6)

In [11]:
fin_service_df.head(60)

,date,sector,weighted_sector_returns,n_stocks_contributing,sector_index,sector_volume
0,2017-01-03,financial_services,0.000000,20.0,100.000000,48125518.0
1,2017-01-04,financial_services,0.353432,20.0,100.353432,26705191.0
2,2017-01-05,financial_services,-0.182223,20.0,100.170565,66991076.0
3,2017-01-06,financial_services,1.397196,20.0,101.570144,97521276.0
4,2017-01-09,financial_services,3.645896,20.0,105.273286,99013315.0
5,2017-01-10,financial_services,-1.454119,20.0,103.742486,135855996.0
6,2017-01-11,financial_services,0.084153,20.0,103.829789,51618427.0
7,2017-01-12,financial_services,1.005168,20.0,104.873453,71053936.0
8,2017-01-13,financial_services,0.968252,20.0,105.888892,49166944.0
9,2017-01-16,financial_services,1.840726,20.0,107.838017,75190170.0


In [ ]:
if len(data) > 400:
    ticker = os.path.splitext(csv_data)[0] ## Captures the name of the stock
    print()
    print(f'{csv_data} engineering begins')

    ## Prepare the data
    print("preparing data")
    prepared_data=prepare_data(data,drop_items=["high_price", "low_price"])

    ## Implement stock returns (%)
    print("stock returns")
    data_with_stock_returns=stock_returns_percent(prepared_data)

    ## Implement lagging
    print("lag returns")
    data_lag_returns=lag_returns(data_with_stock_returns,lag_items=[1,5,10])

    ## Applying Moving Averege
    print("moving_averages")
    data_moving_average=moving_averages(data_lag_returns,averaging_items=[5,20,50])

    ## Applying Rolling Volatility
    print("rolling_volatility")
    data_rolling_volatility = rolling_volatility(data_moving_average,volatility_items=[20,50])

    ## Intrducing RSI
    print("relative_strength_index")
    data_rsi=relative_strength_index(data_rolling_volatility,rsi_length=14)

    ## Calculating the MACD
    print("ma_convergence_divergence")
    data_macd=ma_convergence_divergence(data_rsi)

    ## Get stock volume moving average
    print("volume_moving_average")
    data_volume_ma = volume_moving_average(data_macd,volume_items=[20])

    ## Clean and save data
    print("clean_and_save")
    clean_and_save(data_volume_ma,stock_name=ticker)

    print(f"{ticker} engineering completed")

In [ ]:
## let me keep this here
def weighted_stock_returns(csv_data:pd.DataFrame, ticker:str, sector_market_cap:float, all_stocks:pd.DataFrame) -> dict:
    try:
        stock_mkt_cap = get_market_cap(ticker, all_stocks)
        weight=stock_mkt_cap/sector_market_cap
        csv_data['weighted_stock_return']=csv_data['returns_pct']*weight
    except Exception as e:
        print('Error: ', e)
    return csv_data


## This will aggregate the weighted return and put in a dataframe
def weighted_sector_stock_return(weighted_stocks_returns_dict:dict, sector_df:pd.DataFrame) -> pd.DataFrame:
    # Concatenate all dataframes, keeping only date + weighted_returns
    combined = pd.concat(
        [df[['trade_date', 'weighted_stock_return']] for df in weighted_stocks_returns_dict.values()],
        ignore_index=True
    )

    # Sum weighted_returns per date across all stocks
    daily_totals = combined.groupby('trade_date')['weighted_stock_return'].sum()

    # Map onto sector_df based on matching dates
    sector_df['weighted_sector_returns'] = sector_df['date'].map(daily_totals)

    return sector_df